**2. Preparar el entorno y cargar el dataset

In [1]:
# Importar librerías necesarias
import os
import requests
import pandas as pd
import pathlib

**2.1 Cargar el dataset

In [2]:
# Define el directorio donde se guardarán los datos
_data_root = './data/covertype'
# Define la ruta completa del archivo de entrenamiento
_data_filepath = os.path.join(_data_root, 'covertype_train.csv')

In [3]:
# Crea el directorio si no existe
os.makedirs(_data_root, exist_ok=True)

In [4]:
# Si el archivo no existe, se descarga desde la URL (reemplaza la URL con la que corresponda)
if not os.path.isfile(_data_filepath):
    #https://archive.ics.uci.edu/ml/machine-learning-databases/covtype/ 
    url = "https://docs.google.com/uc?export=download&id=1lVF1BCWLH4eXXV_YOJzjR7xZjj-wAGj9"  # Ajusta este enlace según tu necesidad
    r = requests.get(url, allow_redirects=True, stream=True)
    with open(_data_filepath, 'wb') as f:
        f.write(r.content)
    print("Dataset descargado.")
else:
    print("El dataset ya existe.")

El dataset ya existe.


In [5]:
# Cargar el dataset en un DataFrame
df = pd.read_csv(_data_filepath)
print("Dimensiones del dataset:", df.shape)
# Mostrar las primeras 5 filas
df.head()

Dimensiones del dataset: (116203, 13)


,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Wilderness_Area,Soil_Type,Cover_Type
0,2991,119,7,67,11,1015,233,234,133,1570,Commanche,C7202,1
1,2876,3,18,485,71,2495,192,202,144,1557,Commanche,C7757,1
2,3171,315,2,277,9,4374,213,237,162,1052,Rawah,C7745,0
3,3087,342,13,190,31,4774,193,221,166,752,Rawah,C7745,0
4,2835,158,10,212,41,3596,231,242,141,3280,Rawah,C4744,1


In [6]:
# Mostrar información general del dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116203 entries, 0 to 116202
Data columns (total 13 columns):
 #   Column                              Non-Null Count   Dtype 
---  ------                              --------------   ----- 
 0   Elevation                           116203 non-null  int64 
 1   Aspect                              116203 non-null  int64 
 2   Slope                               116203 non-null  int64 
 3   Horizontal_Distance_To_Hydrology    116203 non-null  int64 
 4   Vertical_Distance_To_Hydrology      116203 non-null  int64 
 5   Horizontal_Distance_To_Roadways     116203 non-null  int64 
 6   Hillshade_9am                       116203 non-null  int64 
 7   Hillshade_Noon                      116203 non-null  int64 
 8   Hillshade_3pm                       116203 non-null  int64 
 9   Horizontal_Distance_To_Fire_Points  116203 non-null  int64 
 10  Wilderness_Area                     116203 non-null  object
 11  Soil_Type                           116

**3. Selección de características

In [7]:
from dataclasses import dataclass
from typing import List
from pathlib import Path

data_root_prepro = Path("./data_prepro")
data_root_prepro.mkdir(parents=True, exist_ok=True)  # Crea el directorio si no existe

@dataclass
class DataConfig:
    target_col: str
    non_numeric_cols: List[str]
    final_df_path: Path
    
# Definir data_root_prepro y crear el directorio si es necesario
data_root_prepro = Path("./data_prepro")
data_root_prepro.mkdir(parents=True, exist_ok=True)

    
#Creating an instance with specific values
config = DataConfig(
    target_col="Cover_Type",
    non_numeric_cols=list(df.select_dtypes(include=['object']).columns),
    final_df_path= data_root_prepro / "covertype_preprocessed.csv"
)

La ejecución de la siguiente celda se omite mediante el comando %%script false --no-raise-error, ya que contiene la normalización de lso datos, un proceso que ya se realizó previamente según lo indicado en el documento.

**"Recuerde que primero debe preparar las características de entrada y de destino:"

Sin embargo, más adelante en el documento se asume que los datos conservan sus valores originales, por lo que la normalización se aplica posteriormente utilizando las herramientas de TFX.

In [8]:
%%script false --no-raise-error

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

# Drop non-numeric columns
df_1 = df.drop(columns=config.non_numeric_cols)

#Separate features and label
X = df_1.drop(columns=[config.target_col])
Y = df_1[config.target_col].astype('category')

#Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#Convert back to Dataframe with original column names
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

#Implement f_classif as score function and select the 8 best columns
selector = SelectKBest(score_func=f_classif, k=8)
selector.fit(X,y)

#Create and print a df comparing the column and the result (if its retained or not)
selected_columns_df = pd.DataFrame({
    'Column': X_scaled.columns,
    'Retain': selector.get_support()
})
selected_columns_df

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

#Drop non-numeric columns
df_1 = df.drop(columns=config.non_numeric_cols)

#Separate features and label
X = df_1.drop(columns=[config.target_col])
y = df_1[config.target_col].astype('category')

#Implement f_classif as score function and select the 8 best columns
selector = SelectKBest(score_func=f_classif, k=8)
selector.fit(X,y)

#Select the best features using boolean mask
X_selected = X.loc[:, selector.get_support()]

#Create and print a df comparing the column and the result (if its retained or not)
selected_columns_df = pd.DataFrame({
    'Column': X.columns,
    'Retain': selector.get_support()
})
selected_columns_df

,Column,Retain
0,Elevation,True
1,Aspect,False
2,Slope,True
3,Horizontal_Distance_To_Hydrology,True
4,Vertical_Distance_To_Hydrology,True
5,Horizontal_Distance_To_Roadways,True
6,Hillshade_9am,True
7,Hillshade_Noon,True
8,Hillshade_3pm,False
9,Horizontal_Distance_To_Fire_Points,True


In [10]:
# Add the target column back

final_df = X_selected.copy()
final_df[config.target_col] = y.values

#Save the updated dataframe to csv
final_df.to_csv(config.final_df_path, index=False)

In [11]:
final_df.head()

,Elevation,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Horizontal_Distance_To_Fire_Points,Cover_Type
0,2991,7,67,11,1015,233,234,1570,1
1,2876,18,485,71,2495,192,202,1557,1
2,3171,2,277,9,4374,213,237,1052,0
3,3087,13,190,31,4774,193,221,752,0
4,2835,10,212,41,3596,231,242,3280,1


Nota: Se debe tener cargado el dataset (final_df) en memoria, pues posteriormente se debe hacer una división para algunas pruebas.

**4. Data PipeLine

**4.1 Configurar el contexto interactivo

In [14]:
# Instala TFX (si aún no lo has hecho)
!pip install tfx

import os
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext

# Define la raíz del pipeline (por ejemplo, en una carpeta en el directorio actual)
pipeline_root = os.path.join(os.getcwd(), "tfx_pipeline_output")

# Crea el contexto interactivo
context = InteractiveContext(pipeline_root=str(pipeline_root))

# Verifica el directorio
print("Pipeline root:", pipeline_root)

Pipeline root: /home/jovyan/work/tfx_pipeline_output


**Generando ejemplos

In [16]:
from tfx.components import CsvExampleGen
import os

# Define la ruta de tus datos (ajústala según corresponda)
data_root_prepro = os.path.join(os.getcwd(), "data", "preprocessed")

# Instancia CsvExampleGen usando la carpeta que contiene el dataset en formato CSV
example_gen = CsvExampleGen(input_base=str(data_root_prepro))

# Ejecuta el componente en el contexto interactivo (asumiendo que ya definiste 'context')
context.run(example_gen)

print("CsvExampleGen ok")

RuntimeError: Split pattern /home/jovyan/work/data/preprocessed/* does not match any files.

**4.3 Estadísticas

In [ ]:
# get the artifact object
artifact = example_gen.outputs('examples').get()[0]

# print split names and uri
print(f'split names: (artifact.split_names)')
print(f'artifact uri: (artifact.uri)')